# Evaluation and offline reports
The primary comparison uses no-hint fully-correct rate on the unchanged, previously explored benchmark. Current-law reviews stay separate. No improvement is claimed before complete compatible paired runs. Core behavior: specs 06, 07 and 09.


In [ ]:
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CORPUS = PROJECT / "data/corpora/current-final"
INDEX = PROJECT / "data/indexes/current-bge"
BENCHMARK = PROJECT / "data/evaluation"
RUN_DIRS = []  # Add explicit Path objects for saved evaluation runs.
DIAGNOSTIC_DIRS = [PROJECT / "data/diagnostics/current-final"]
SAVED_REPORT = PROJECT / "data/reports/current-final"
NEW_RUN = PROJECT / "data/runs/notebook-no-rag"
REPORT = PROJECT / "data/reports/notebook-report"
RUN_EVALUATION = False
WRITE_REPORT = False


## 1. Inspect the frozen benchmark and saved runs
Original answers and references are evaluation inputs only. Missing scores remain missing, never fabricated zeros.


In [ ]:
from legal_rag.evaluation import load_benchmark, load_results, compare_runs
from legal_rag.artifacts import read_json

benchmark = load_benchmark(BENCHMARK)
print({"intents": len(benchmark.questions), "view": benchmark.view, "warnings": benchmark.warnings})
for directory in RUN_DIRS:
    print(read_json(directory / "metrics.json"))
    print(load_results(directory)[0].to_json_record())
if len(RUN_DIRS) > 1:
    print(compare_runs(RUN_DIRS))
if (SAVED_REPORT / "manifest.json").is_file():
    print({"saved_report": str(SAVED_REPORT / "index.html")})
    print(read_json(SAVED_REPORT / "manifest.json")["warnings"])
else:
    print("The delivered report is not available yet at the configured path.")


## 2. Optional evaluation
Disabled by default. Enabling this dispatches generation/judge/control API calls through the same CLI module, after credential, current pricing and budget checks. Edit provider/model choices and use a new output directory. No API call is required to inspect saved results.


In [ ]:
if RUN_EVALUATION:
    from legal_rag.cli import main

    exit_code = main(["evaluate", "--profile", "no_rag", "--provider", "utopia",
        "--corpus", str(CORPUS), "--index", str(INDEX), "--benchmark", str(BENCHMARK),
        "--output", str(NEW_RUN), "--local-files-only"])
    print({"exit_code": exit_code})


## 3. Optional offline report
This writes HTML from saved artifacts and calls no model. Without evaluation runs, the report contains the corpus and saved retrieval diagnostics only. Existing output directories are rejected.


In [ ]:
if WRITE_REPORT:
    from legal_rag.reporting import build_report

    print(build_report(CORPUS, RUN_DIRS, REPORT, diagnostic_dirs=DIAGNOSTIC_DIRS))
